In [ ]:
!pip install datasets[audio] librosa transformers

In [ ]:
from datasets import load_from_disk, concatenate_datasets, Audio, disable_caching

import sys
import glob
import pickle
import random
import numpy as np
from tqdm.notebook import tqdm
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
from torch.nn import Module, Linear
from torchvision.models import resnet18

from google.colab import drive


NUM_FOLDS = 3
NUM_CLASSES = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42

In [ ]:
drive.mount('/content/drive')

In [ ]:
%cd "/content/drive/MyDrive/Colab Notebooks/ASP/Project/speaker_chunks/resnet_contr_cross_val_6e_4s"

In [ ]:
def seed_everything(seed=42):
    # Python
    random.seed(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch CPU
    torch.manual_seed(seed)

    # PyTorch GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    # CuDNN
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)

# Data Preprocessing


Load data

In [ ]:
# Disable caching so no additional files are saved in drive
disable_caching()

In [ ]:
drive_path = "/content/drive/MyDrive/Colab Notebooks/ASP/Project/speaker_chunks/chunk_*"
chunk_paths = sorted(glob.glob(drive_path))

chunks = []
for path in chunk_paths:
    ds = load_from_disk(path)
    ds = ds.cast_column("audio_path", Audio(sampling_rate=16000))
    chunks.append(ds)

full_dataset = concatenate_datasets(chunks)

In [ ]:
split = full_dataset.train_test_split(test_size=0.1, seed=42)
trainval_dataset = split["train"] # For cross-validation

# Get the labels for stratification
labels = np.array(trainval_dataset["label"])

# 9-way stratified split, take 3 folds
skf = StratifiedKFold(n_splits=9, shuffle=True, random_state=42)
fold_indices = list(skf.split(np.zeros(len(labels)), labels))

folds = [] # Collect the 3 folds
for i in range(NUM_FOLDS):
    train_idx, dev_idx = fold_indices[i]
    train_dataset = trainval_dataset.select(train_idx.tolist())
    dev_dataset = trainval_dataset.select(dev_idx.tolist())
    folds.append((train_dataset, dev_dataset))
    print(f"Fold {i}: train={len(train_dataset)}")

Prepocess data using the Dataset_Builder class

In [ ]:
# Add specific folder containing dataset_builder.py to path
sys.path.append("/content/drive/MyDrive/Colab Notebooks/ASP/Project/speaker_chunks/contrastive/files/utils")

from dataset_builder import Dataset_Builder

In [ ]:
# Training sets
config = {"shortest_duration": 4.0, "timesteps": 400} # (optional)
dataset_train_folds = []

# Loop for 3 folds
for fold in folds:
  train_builder = Dataset_Builder(fold[0], **config)
  train_builder.filter()
  train_builder.preprocess()
  dataset_train_folds.append(train_builder.get_dataset())

# Post Training

Load precomputed spk2indxs (from training run)

In [ ]:
spk2indxs_train_folds = []
for indx in range(1, len(dataset_train_folds)+1):
  with open(f"spk2indxs_train_fold_{indx}.pkl", "rb") as f:
    spk2indxs_train = pickle.load(f)
  spk2indxs_train_folds.append(spk2indxs_train)

Define Model

In [ ]:
class ContrastiveModel(Module):
  def __init__(self, base_model, embed_dim):
    super().__init__()
    self.model = base_model
    self.model.fc = Linear(in_features=512, out_features=embed_dim, bias=True)

  def forward(self, x):
    x = self.model(x)
    return nn.functional.normalize(x, dim=-1) # Avoid representation collapse

Compute Centroids

In [ ]:
fold_centroids = []
for fold_idx, (dataset_train, spk2indxs_train) in enumerate(zip(dataset_train_folds, spk2indxs_train_folds)):

  print(f"\n=========== FOLD {fold_idx + 1} ===========")

  # Re-initialize everything per fold
  base_model = resnet18(weights="DEFAULT")
  model = ContrastiveModel(base_model, NUM_CLASSES)
  model.load_state_dict(torch.load(f"best_speaker_resnet_fold{fold_idx+1}.pth", map_location=DEVICE, weights_only=True))
  model.to(DEVICE)
  model.eval()

  items = spk2indxs_train.items()

  centroids = {}
  with torch.no_grad():
    for spk, indxs in tqdm(items, desc="Speakers", total=len(items)):
      centroid = np.zeros(NUM_CLASSES)
      for indx in tqdm(indxs, desc=f"Calculating Speaker {spk} Centroid", total=len(indxs)):
        log_mel = dataset_train[indx]["log_mel"].unsqueeze(0).to(DEVICE)
        embed = model(log_mel)
        embed = embed.squeeze().detach().cpu().numpy()
        centroid += embed
      centroid /= len(indxs)
      centroids[spk] = centroid

  fold_centroids.append(centroids)

Save centroids

In [ ]:
for indx, centroids_dict in enumerate(fold_centroids):
  string_keyed_centroids = {str(key): value for key, value in centroids_dict.items()}
  np.savez_compressed(f"centroids_train_fold_{indx+1}.npz", **string_keyed_centroids)

Test import

In [ ]:
centroids_loaded = np.load("centroids_train_fold_1.npz")
centroids_restored = {key: centroids_loaded[key] for key in centroids_loaded.files}

print(centroids_restored)